# Region Value Analysis
This notebook analyzes the unique Region values in survey dataframes while respecting the existing environment.

In [1]:
import os
from secure_data_utils import load_json_with_types
import pandas as pd
import numpy as np

In [2]:
# Load the dictionary with survey data
ruta_enc = '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/encuestas'
los_mex_dict = load_json_with_types(os.path.join(ruta_enc, 'los_mex_dict.json'))

# Get dictionaries
enc_nom_dict = los_mex_dict.get('enc_nom_dict', {})
enc_dict = los_mex_dict.get('enc_dict', {})
print(f"Found {len(enc_nom_dict)} surveys in enc_nom_dict")
print(f"Found {len(enc_dict)} surveys in enc_dict")

Found 26 surveys in enc_nom_dict
Found 26 surveys in enc_dict


In [3]:
# Expected region values based on mapping from ses_analysis.py
expected_values = {1.0, 2.0, 3.0, 4.0, 5.0}
print("Expected Region values:", sorted(expected_values))

Expected Region values: [1.0, 2.0, 3.0, 4.0, 5.0]


In [4]:
# Dictionary to store results
results = {}

# Analyze each survey's dataframe
for survey_name in enc_nom_dict:
    if survey_name not in enc_dict:
        print(f"Warning: Survey {survey_name} not found in enc_dict")
        continue
        
    survey_data = enc_dict[survey_name]
    if not isinstance(survey_data, dict) or 'dataframe' not in survey_data:
        print(f"Warning: Invalid data structure for survey {survey_name}")
        continue
        
    df = survey_data['dataframe']
    if not isinstance(df, pd.DataFrame):
        print(f"Warning: No DataFrame found for survey {survey_name}")
        continue
    
    # Check both 'Region' and 'region' columns
    for col in ['Region', 'region']:
        if col not in df.columns:
            continue
            
        # Convert to float for consistent comparison
        unique_values = df[col].dropna().astype(float).unique()
        unique_values = sorted(unique_values)
        n_unique = len(unique_values)
        n_nulls = df[col].isna().sum()
        
        results[f"{survey_name}_{col}"] = {
            'column': col,
            'n_unique': n_unique,
            'unique_values': unique_values,
            'n_nulls': n_nulls,
            'unexpected_values': [v for v in unique_values 
                               if v not in expected_values]
        }

## Results Analysis

In [5]:
# Print results in a formatted way
print("\nRegion Value Analysis")
print("=" * 80)

for key, data in sorted(results.items()):
    print(f"\nSurvey-Column: {key}")
    print("-" * 40)
    print(f"Number of unique values: {data['n_unique']}")
    print(f"Number of null values: {data['n_nulls']}")
    print(f"Unique values found: {data['unique_values']}")
    if data['unexpected_values']:
        print(f"⚠️  Unexpected values found: {data['unexpected_values']}")
    print()


Region Value Analysis

Survey-Column: CIENCIA_Y_TECNOLOGIA_Region
----------------------------------------
Number of unique values: 4
Number of null values: 0
Unique values found: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]


Survey-Column: CORRUPCION_Y_CULTURA_DE_LA_LEGALIDAD_Region
----------------------------------------
Number of unique values: 4
Number of null values: 0
Unique values found: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]


Survey-Column: CULTURA_CONSTITUCIONAL_region
----------------------------------------
Number of unique values: 4
Number of null values: 0
Unique values found: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]


Survey-Column: CULTURA_LECTURA_Y_DEPORTE_Region
----------------------------------------
Number of unique values: 4
Number of null values: 0
Unique values found: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]


Survey-Column: CULTURA_POLITICA_Region
--

In [6]:
# Summary statistics
print("\nSummary Statistics")
print("=" * 80)
columns_analyzed = len(results)
surveys_with_region = len({k.split('_')[0] for k in results.keys()})
surveys_with_unexpected = len({k.split('_')[0] for k, v in results.items() if v['unexpected_values']})

print(f"\nTotal surveys analyzed: {len(enc_nom_dict)}")
print(f"Surveys with Region data: {surveys_with_region}")
print(f"Total columns analyzed: {columns_analyzed}")
print(f"Surveys with unexpected values: {surveys_with_unexpected}")


Summary Statistics

Total surveys analyzed: 26
Surveys with Region data: 24
Total columns analyzed: 28
Surveys with unexpected values: 0


## Additional Analysis

In [7]:
# Create a distribution of Region values across all surveys
region_dist = {}

for key, data in results.items():
    survey_name = key.split('_')[0]
    col = data['column']
    if col == 'Region':  # Only look at original Region column
        for val in data['unique_values']:
            if val not in region_dist:
                region_dist[val] = set()
            region_dist[val].add(survey_name)

print("\nRegion Value Distribution")
print("=" * 80)
print("\nValue | Number of Surveys | Survey Names")
print("-" * 60)
for val in sorted(region_dist.keys()):
    surveys = region_dist[val]
    print(f"{val:5.1f} | {len(surveys):16d} | {', '.join(sorted(surveys))}")


Region Value Distribution

Value | Number of Surveys | Survey Names
------------------------------------------------------------
  1.0 |               23 | CIENCIA, CORRUPCION, CULTURA, DERECHOS, ECONOMIA, EDUCACION, ENVEJECIMIENTO, FAMILIA, FEDERALISMO, GENERO, GLOBALIZACION, IDENTIDAD, INDIGENAS, JUEGOS, LA, MEDIO, MIGRACION, NINOS, POBREZA, RELIGION, SALUD, SEGURIDAD, SOCIEDAD
  2.0 |               23 | CIENCIA, CORRUPCION, CULTURA, DERECHOS, ECONOMIA, EDUCACION, ENVEJECIMIENTO, FAMILIA, FEDERALISMO, GENERO, GLOBALIZACION, IDENTIDAD, INDIGENAS, JUEGOS, LA, MEDIO, MIGRACION, NINOS, POBREZA, RELIGION, SALUD, SEGURIDAD, SOCIEDAD
  3.0 |               23 | CIENCIA, CORRUPCION, CULTURA, DERECHOS, ECONOMIA, EDUCACION, ENVEJECIMIENTO, FAMILIA, FEDERALISMO, GENERO, GLOBALIZACION, IDENTIDAD, INDIGENAS, JUEGOS, LA, MEDIO, MIGRACION, NINOS, POBREZA, RELIGION, SALUD, SEGURIDAD, SOCIEDAD
  4.0 |               23 | CIENCIA, CORRUPCION, CULTURA, DERECHOS, ECONOMIA, EDUCACION, ENVEJECIMIENTO, FAMI

In [12]:
# Find surveys that have Region value 5
surveys_with_5 = set()
details_with_5 = []

# Direct lookup since we're using the actual survey name
for key, data in results.items():
    if 5.0 in data['unique_values']:
        # Split on last underscore since survey names might contain underscores
        parts = key.split('_')
        survey_name = '_'.join(parts[:-1])  # Everything except last part
        col_name = parts[-1]  # Last part is column name
            
        # Get more details about this survey
        df = enc_dict[survey_name]['dataframe']
        if col_name in df.columns:
            region_counts = df[col_name].value_counts()
            n_region_5 = region_counts.get(5.0, 0)
            total_regions = len(df)
            details_with_5.append({
                'survey': survey_name,
                'count_region_5': n_region_5,
                'total_regions': total_regions,
                'percentage': (n_region_5/total_regions) * 100
            })
            surveys_with_5.add(survey_name)

print(f"\nFound {len(surveys_with_5)} surveys with Region value 5:")
print("=" * 80)
for details in sorted(details_with_5, key=lambda x: x['percentage'], reverse=True):
    print(f"\nSurvey: {details['survey']}")
    print(f"Count of Region=5: {details['count_region_5']:,}")
    print(f"Total regions: {details['total_regions']:,}")
    print(f"Percentage: {details['percentage']:.1f}%")
    
# Also show the mapping to friendly names
print("\nFriendly names:")
print("=" * 80)
for survey in sorted(surveys_with_5):
    if survey in enc_nom_dict:
        print(f"{survey} -> {enc_nom_dict[survey]}")


Found 1 surveys with Region value 5:

Survey: JUEGOS_DE_AZAR
Count of Region=5: 260
Total regions: 1,200
Percentage: 21.7%

Survey: JUEGOS_DE_AZAR
Count of Region=5: 260
Total regions: 1,200
Percentage: 21.7%

Friendly names:
JUEGOS_DE_AZAR -> JUE


In [10]:
# Check survey name mappings
print("enc_nom_dict:")
for k, v in enc_nom_dict.items():
    print(f"{k}: {v}")
    
print("\nKeys in results dict:")
print(sorted(results.keys()))

enc_nom_dict:
IDENTIDAD_Y_VALORES: IDE
MEDIO_AMBIENTE: MED
POBREZA: POB
CULTURA_POLITICA: CUL
RELIGION_SECULARIZACION_Y_LAICIDAD: REL
SEGURIDAD_PUBLICA: SEG
SALUD: SAL
INDIGENAS: IND
SOCIEDAD_DE_LA_INFORMACION: SOC
ENVEJECIMIENTO: ENV
DERECHOS_HUMANOS_DISCRIMINACION_Y_GRUPOS_VULNERABLES: DER
CORRUPCION_Y_CULTURA_DE_LA_LEGALIDAD: COR
LA_CONDICION_DE_HABITABILIDAD_DE_VIVIENDA_EN_MEXICO: HAB
GLOBALIZACION: GLO
JUSTICIA: JUS
JUEGOS_DE_AZAR: JUE
MIGRACION: MIG
FEDERALISMO: FED
GENERO: GEN
CULTURA_CONSTITUCIONAL: CON
CULTURA_LECTURA_Y_DEPORTE: DEP
ECONOMIA_Y_EMPLEO: ECO
NINOS_ADOLESCENTES_Y_JOVENES: NIN
FAMILIA: FAM
CIENCIA_Y_TECNOLOGIA: CIE
EDUCACION: EDU

Keys in results dict:
['CIENCIA_Y_TECNOLOGIA_Region', 'CORRUPCION_Y_CULTURA_DE_LA_LEGALIDAD_Region', 'CULTURA_CONSTITUCIONAL_region', 'CULTURA_LECTURA_Y_DEPORTE_Region', 'CULTURA_POLITICA_Region', 'DERECHOS_HUMANOS_DISCRIMINACION_Y_GRUPOS_VULNERABLES_Region', 'ECONOMIA_Y_EMPLEO_Region', 'EDUCACION_Region', 'ENVEJECIMIENTO_Region', 'FAMILI